# Lab Solutions: Web Scraping + File I/O


## Basic

The basic version only uses the people who are present in the HTML when the page first loads. We do not click **"Load more"** here.

For each card, we follow the profile link and collect:

- Name
- Title
- E-mail
- Web page
- Specialization

Some profiles do not contain every field, so those values are left blank.


In [1]:
from bs4 import BeautifulSoup
import urllib.request
import csv
import time
import os
import ssl
import certifi


In [4]:
# Use certifi's certificate file when urllib opens an HTTPS page.
# The context has to be passed to urlopen() to actually be used.
context = ssl.create_default_context(cafile=certifi.where())

people_url = 'https://polisci.washu.edu/people?cat=88&letter=All'

# Basic: read the page once. This only gives us the cards currently loaded.
page = urllib.request.urlopen(people_url, context=context)
soup = BeautifulSoup(page.read(), 'html.parser')

cards = soup.find_all('a', {'class': 'card'})

print("Cards on the first page:", len(cards))


Cards on the first page: 24


In [3]:
with open('lab02_basic.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(
        f,
        fieldnames=("name", "title", "email", "website", "specialization")
    )
    w.writeheader()

    for c in range(len(cards)):
        print("Working on " + str(c + 1) + " of " + str(len(cards)) + ":")

        try:
            fac = {}

            # Name and title come from the card on the people page.
            fac['name'] = ' '.join(
                cards[c].find('h3').get_text(" ", strip=True).split()
            )

            try:
                fac['title'] = cards[c].find(
                    'div',
                    {'class': 'dept'}
                ).get_text(" ", strip=True)
            except:
                fac['title'] = ''

            # Build the URL for the person's profile page.
            interior = (
                'https://polisci.washu.edu' + cards[c]['href']
            )

            interior_page = urllib.request.urlopen(
                interior,
                context=context
            )

            interior_soup = BeautifulSoup(
                interior_page.read(),
                'html.parser'
            )

            # Not every profile has every field, so missing values are blank.
            try:
                fac['email'] = interior_soup.find(
                    'ul',
                    {'class': 'detail contact'}
                ).find('a').get_text(strip=True)
            except:
                fac['email'] = ''

            try:
                fac['website'] = interior_soup.find(
                    'ul',
                    {'class': 'links'}
                ).find('a')['href']
            except:
                fac['website'] = ''

            try:
                fac['specialization'] = interior_soup.find(
                    'div',
                    {'class': 'post-excerpt'}
                ).get_text(" ", strip=True)
            except:
                fac['specialization'] = ''

            w.writerow(fac)

        except Exception as e:
            print("FAILED:", e)
            continue

        # Small pause so we do not hit the site too quickly.
        time.sleep(1)

print("All done!")


Working on 1 of 24:
Working on 2 of 24:
Working on 3 of 24:
Working on 4 of 24:
Working on 5 of 24:
Working on 6 of 24:
Working on 7 of 24:
Working on 8 of 24:
Working on 9 of 24:
Working on 10 of 24:
Working on 11 of 24:
Working on 12 of 24:
Working on 13 of 24:
Working on 14 of 24:
Working on 15 of 24:
Working on 16 of 24:
Working on 17 of 24:
Working on 18 of 24:
Working on 19 of 24:
Working on 20 of 24:
Working on 21 of 24:
Working on 22 of 24:
Working on 23 of 24:
Working on 24 of 24:
All done!


## Challenge: Load More + Deduplicate

The challenge has two extra steps.

First, Selenium opens the page and repeatedly clicks **"Load more."** We wait after each click so the next batch of cards has time to appear. Only after the page is fully loaded do we pass the final HTML to BeautifulSoup.

Second, we remove duplicates using each person's profile URL. If the same `href` appears more than once, we keep only the first card.

After that, the profile-page scraping is the same as in the Basic solution.


In [ ]:
# Run this once if Selenium is not installed in your environment:
# !pip install selenium

from selenium import webdriver
from selenium.webdriver.common.by import By


In [ ]:
# Open the page in a real browser because we need to click a button.
driver = webdriver.Chrome()
driver.get('https://polisci.washu.edu/people?cat=88&letter=All')

time.sleep(3)

# Keep looking for a "Load more" button.
# find_elements() returns an empty list when the button is gone.
while True:
    load_more_buttons = driver.find_elements(
        By.XPATH,
        "//*[self::a or self::button]"
        "[contains(normalize-space(.), 'Load more')]"
    )

    if len(load_more_buttons) == 0:
        print("No more Load more button.")
        break

    load_more = load_more_buttons[0]
    print("Clicking Load more...")

    # Move the button into view and click it.
    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        load_more
    )

    time.sleep(1)

    driver.execute_script(
        "arguments[0].click();",
        load_more
    )

    # Give the next group of cards time to load.
    time.sleep(3)

# Get the HTML only after all available cards have been loaded.
html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, 'html.parser')
cards = soup.find_all('a', {'class': 'card'})

print("Cards before removing duplicates:", len(cards))


In [ ]:
# The same person can appear more than once after loading more results.
# The profile URL is a simple way to identify duplicates.
unique_cards = []
seen = set()

for card in cards:
    href = card.get('href')

    if href and href not in seen:
        seen.add(href)
        unique_cards.append(card)

cards = unique_cards

print("Total unique people found:", len(cards))


In [ ]:
with open('lab02_challenge.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(
        f,
        fieldnames=("name", "title", "email", "website", "specialization")
    )
    w.writeheader()

    for c in range(len(cards)):
        print("Working on " + str(c + 1) + " of " + str(len(cards)) + ":")

        try:
            fac = {}

            fac['name'] = ' '.join(
                cards[c].find('h3').get_text(" ", strip=True).split()
            )

            try:
                fac['title'] = cards[c].find(
                    'div',
                    {'class': 'dept'}
                ).get_text(" ", strip=True)
            except:
                fac['title'] = ''

            interior = (
                'https://polisci.washu.edu' + cards[c]['href']
            )

            interior_page = urllib.request.urlopen(
                interior,
                context=context
            )

            interior_soup = BeautifulSoup(
                interior_page.read(),
                'html.parser'
            )

            try:
                fac['email'] = interior_soup.find(
                    'ul',
                    {'class': 'detail contact'}
                ).find('a').get_text(strip=True)
            except:
                fac['email'] = ''

            try:
                fac['website'] = interior_soup.find(
                    'ul',
                    {'class': 'links'}
                ).find('a')['href']
            except:
                fac['website'] = ''

            try:
                fac['specialization'] = interior_soup.find(
                    'div',
                    {'class': 'post-excerpt'}
                ).get_text(" ", strip=True)
            except:
                fac['specialization'] = ''

            w.writerow(fac)

        except Exception as e:
            print("FAILED:", e)
            continue

        time.sleep(1)

print("All done!")
